# 06 · Normalização (LayerNorm)

**Peça nova:** `LayerNorm`. No notebook 05 o fluxo residual crescia de bloco em bloco, e cada bloco recebia entradas numa escala diferente. A LayerNorm normaliza **cada vetor de token** para média 0 e variância 1 e depois aplica uma escala (γ) e um deslocamento (β) aprendidos.

Onde colocá-la:
- **post-norm** (Transformer original, 2017): `x = LN(x + f(x))`. A normalização fica no caminho do residual.
- **pre-norm** (GPT-2 em diante): `x = x + f(LN(x))`. O residual fica limpo; normaliza-se só a entrada de cada subcamada, e uma LN final antes da saída.

Daqui em diante usamos o modelo de referência `lab.model.GPT`, que monta tudo o que você construiu até aqui e deixa cada peça ser ligada ou desligada por `GPTConfig`. Diário: `F1-NRM`.

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from lab.data import get_batch, shakespeare
from lab.model import GPT, Block, GPTConfig, count_params
from lab.plot import compare, table
from lab.train import pick_device, train

DEVICE = pick_device()
ds = shakespeare()
tok, V = ds.tok, ds.tok.vocab_size
BLOCK, D, STEPS, LR = 32, 64, 3000, 1e-3
print(ds, "| dispositivo:", DEVICE)

## A peça: `LayerNorm` à mão

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)                 # média de cada token (sobre as d_model dimensões)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

### Testes
1. Igual à `nn.LayerNorm` do PyTorch.
2. Cada token sai com média 0 e desvio 1, seja qual for a escala de entrada. Repare que ela normaliza **cada linha** (token), não cada coluna (como a BatchNorm).

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 5, D) * 7 + 3
print("igual à nn.LayerNorm:", torch.allclose(LayerNorm(D)(x), nn.LayerNorm(D)(x), atol=1e-5))
y = LayerNorm(D)(x)
for nome, t in [("entrada", x), ("saída", y)]:
    print(f"{nome:8} token 0: média {t[0, 0].mean():6.2f}, desvio {t[0, 0].std(unbiased=False):5.2f}")

## O bloco de referência
Leia o `Block` de `lab/model.py`: é o seu bloco do notebook 05 com a opção `norm`. A lógica toda está em `_sublayer`.

In [ ]:
print(inspect.getsource(Block))

Uma diferença em relação ao notebook 05: o `GPT` inicializa os pesos com desvio 0,02 (como o GPT-2), em vez do padrão do PyTorch. Por isso compare as execuções deste notebook entre si.

## Treino de referência: sem norma, pre-norm e post-norm (4 blocos)

**Minha previsão:** _(qual das três chega mais baixo? alguma é muito mais lenta no começo?)_

In [ ]:
def make(**kw):
    cfg = GPTConfig(vocab_size=V, block_size=BLOCK, d_model=D, n_layer=4, n_head=4, **kw)
    torch.manual_seed(1337)
    return GPT(cfg)

models = {}
for norm in ["none", "pre", "post"]:
    models[norm] = make(norm=norm)
    run = train(models[norm], ds, name=f"06-norm-{norm}", steps=STEPS, lr=LR, device=DEVICE, verbose=False)
    print(run)
compare(prefix="06-norm", title="Normalização (4 blocos, lr 1e-3)")
table(prefix="06-norm")

---
## Experimentos

### 1. Learning rate alto
A normalização costuma permitir learning rates maiores. Rode sem norma e com pre-norm em 3e-3 e 1e-2.

**Minha previsão:** _(sem norma diverge em algum deles? qual lr é o melhor para cada um?)_

In [ ]:
for lr in [3e-3, 1e-2]:
    for norm in ["none", "pre"]:
        r = train(make(norm=norm), ds, name=f"06-lr{lr:g}-{norm}", steps=STEPS, lr=lr, device=DEVICE, verbose=False, sample_tokens=0)
        print(r)
compare(["06-norm-none", "06-norm-pre"] + [f"06-lr{lr:g}-{n}" for lr in [3e-3, 1e-2] for n in ["none", "pre"]],
        train=False, title="lr 1e-3, 3e-3 e 1e-2, sem norma vs. pre-norm")

### 2. Mais fundo: 8 blocos
**Minha previsão:** _(a diferença entre sem norma e pre-norm aumenta com 8 blocos?)_

In [ ]:
for norm in ["none", "pre"]:
    torch.manual_seed(1337)
    m = GPT(GPTConfig(vocab_size=V, block_size=BLOCK, d_model=D, n_layer=8, n_head=4, norm=norm))
    print(train(m, ds, name=f"06-L8-{norm}", steps=STEPS, lr=LR, device=DEVICE, verbose=False, sample_tokens=0))
compare(prefix="06-L8", title="8 blocos")
table(prefix="06-L8")

### 3. A escala do fluxo residual, com e sem norma
Mesma medição do notebook 05, nos modelos treinados do treino de referência.

**Minha previsão:** _(com pre-norm o fluxo residual para de crescer? lembre que a norma age na entrada de cada subcamada, não no fluxo)_

In [ ]:
def activation_std(model):
    stds = []
    hooks = [b.register_forward_hook(lambda mod, inp, out: stds.append(out.std().item())) for b in model.blocks]
    model.eval()
    torch.manual_seed(0)
    with torch.no_grad():
        model(get_batch(ds.val, 32, BLOCK, next(model.parameters()).device)[0])
    for h in hooks:
        h.remove()
    return stds

for norm, m in models.items():
    plt.plot(activation_std(m), "o-", label=norm)
plt.xlabel("bloco")
plt.ylabel("desvio padrão na saída do bloco")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## Para o diário (`F1-NRM-01`)

- Qual posição da norma funcionou melhor aqui? A diferença foi maior com lr alto ou com 8 blocos?
- O que o gráfico de escala mostrou? Com pre-norm o fluxo ainda cresce; por que isso não atrapalha as subcamadas?
- Com isso você tem todas as peças de um GPT. Faça a lista de cada uma e do ganho que trouxe nos seus experimentos.

**Próximo:** [07 · Shakespeare](07_shakespeare.ipynb), o GPT completo num treino maior.